# Work with protein interfaces

Start from a selected interface ID and rebuild its two-chain coordinate view. Use [Query the release tables](2_query_filter_index.ipynb) to select interfaces and {ref}`Protein interfaces <protein-interface-reference>` for the column definitions.

In [ ]:
import os

os.environ.setdefault("PLINDER_RELEASE", "2026-07")
os.environ.setdefault("PLINDER_RELEASE_NUMBER", "1")

In [ ]:
import biotite.structure as struc
import pandas as pd

from plinder.core import PlinderInterface

## Rebuild one interface

`PlinderInterface` expands the two selected assembly-chain instances from the source mmCIF revision recorded by the release.

In [ ]:
interface = PlinderInterface(system_id="7cm8__1__1.A--2.A")
interface

In [ ]:
pd.DataFrame(
    {
        "chain": interface.chains,
        "full_sequence_length": [
            len(interface.sequences[chain]) for chain in interface.chains
        ],
        "resolved_atom_count": [
            len(interface.chain_structures[chain]) for chain in interface.chains
        ],
    }
)

`sequences` contains the full deposited polymer sequences; coordinate arrays contain only resolved atoms.

## Interface residues and coordinate views

The residue masks are defined over the complete two-chain `atom_array`. Their keys match the assembly-chain IDs in `interface_annotations`.

In [ ]:
summary = []
for chain, mask in interface.interface_residue_masks.items():
    atoms = interface.atom_array[mask]
    summary.append(
        {
            "chain": chain,
            "interface_residues": struc.get_residue_count(atoms),
            "interface_atoms": len(atoms),
        }
    )
pd.DataFrame(summary)

In [ ]:
print(f"Two-chain atoms: {len(interface.atom_array):,}")
print(f"Interface atoms: {len(interface.interface_structure):,}")
print(f"Reconstructed mmCIF: {interface.interface_cif}")

`interface_cif` is a `pathlib.Path`; `atom_array`, the values of `chain_structures`, and `interface_structure` are Biotite `AtomArray` objects. The written mmCIF is self-contained and contains only the selected interface chains.